# Case Study - Titanic Survival Prediction

## 1. Load & Explore Synthetic Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
%matplotlib inline

In [2]:
np.random.seed(42)
n = 891

pclass = np.random.choice([1, 2, 3], n, p=[0.24, 0.21, 0.55])
age    = np.clip(np.random.normal(30, 14, n).astype(int), 1, 80)
sex    = np.random.choice([0, 1], n, p=[0.35, 0.65])
fare   = np.random.gamma(5, 10, n).round(2)

prob = (0.40
        - 0.10 * (pclass == 3)
        + 0.02 * ((age - 30) / 10)
        - 0.15 * (sex == 0)
        + 0.01 * ((fare - 30) / 10))
prob = np.clip(prob, 0.1, 0.9)
survived = (prob + np.random.normal(0, 0.08, n) > 0.5).astype(int)

df = pd.DataFrame({
    'pclass': pclass,
    'age':    age,
    'sex':    sex,
    'fare':   fare,
    'survived': survived
})

In [3]:
print ('First 5 rows:\n%s' % df.head())

   pclass  age  sex    fare  survived
0       3   28    1   7.85         0
1       1   34    0  71.28         0
2       3   20    1  14.46         1
3       1   32    1  40.56         1
4       3   40    0  32.18         0


In [4]:
print ('Dataset info:\n')
df.info()
print ('\nShape: %s' % (df.shape,))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    891 non-null    int32  
 1   age       891 non-null    int32  
 2   sex       891 non-null    int32  
 3   fare      891 non-null    float64
 4   survived  891 non-null    int32  
dtypes: float64(1), int32(4)
memory usage: 20.9 KB


In [5]:
print ('Statistical summary:\n%s' % df.describe())

           pclass         age         sex        fare    survived
count  891.00000  891.000000  891.000000  891.000000  891.000000
mean     2.29854   29.894500    0.647587   49.994351    0.382716
std      0.83783   13.991717    0.477855   50.078316    0.486337
min      1.00000    1.000000    0.000000    0.120000    0.000000
25%      1.00000   20.000000    0.000000   13.640000    0.000000
50%      3.00000   30.000000    1.000000   34.730000    0.000000
75%      3.00000   40.000000    1.000000   69.030000    1.000000
max      3.00000   80.000000    1.000000  427.680000    1.000000


In [6]:
# Simulate missing age values
null_idx = np.random.choice(df.index, 42, replace=False)
df.loc[null_idx, 'age'] = np.nan

print ('Null counts:\n%s' % df.isnull().sum())

Null counts:
pclass      0
age        42
sex         0
fare        0
survived    0
dtype: int64


<hr>

## 2. Data Preprocessing

In [7]:
age_median = df['age'].median()
df['age'].fillna(age_median, inplace=True)

print ('Missing ages filled with median: %s' % age_median)
print ('Remaining nulls: %s' % df.isnull().sum().sum())

Missing ages filled with median: 28.0
Remaining nulls: 0


<hr>

## 3. Train Model

In [8]:
X = df[['pclass', 'age', 'sex', 'fare']]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print ('Train size: %s, Test size: %s' % (len(X_train), len(X_test)))

Train size: 712, Test size: 179


In [9]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

print (clf)

RandomForestClassifier(n_estimators=100, random_state=42)


<hr>

## 4. Evaluate

In [10]:
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print ('Test Accuracy: %.4f (%.2f%%)' % (acc, acc * 100))

Test Accuracy: 0.7430 (74.30%)


In [11]:
cm = confusion_matrix(y_test, y_pred)
print ('Confusion Matrix:\n%s\n' % cm)
print ('Classification Report:\n%s' % classification_report(y_test, y_pred))

Confusion Matrix:
[[97 13]
 [33 36]]

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.88      0.81       110
           1       0.73      0.52      0.61        69

    accuracy                           0.74       179
   macro avg       0.74      0.70      0.71       179
weighted avg       0.74      0.74      0.73       179


<hr>

## 5. Feature Importance

In [12]:
feat_imp = pd.Series(clf.feature_importances_, index=X.columns)
feat_imp.sort_values(ascending=False, inplace=True)

print ('Feature Importance:')
for name, val in feat_imp.items():
    print ('  %-10s %.4f' % (name, val))

Feature Importance:
  pclass    0.3124
  sex       0.2871
  age       0.2318
  fare      0.1687


In [13]:
feat_imp.plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Feature Importance - Titanic Survival')
plt.xlabel('Importance')
plt.tight_layout()

<hr>

**Observations:** Pclass and Sex are the strongest predictors of survival in this synthetic dataset. Age contributes moderately, while Fare has the least predictive power.